In [ ]:
import numpy as np
from matplotlib import pyplot
from sklearn.ensemble import RandomForestClassifier

# input data
# xdf - Pandas dataframe with input features (nraw, ncol) = (nsamples, nfeatures), 
#       e.g., nsamples = number associated sources
# ys - array of true classes (class names used for training)
# nav = 10 - number of trainings (with random splits into training/testing samples) 
#            used for the calculation of the distribution of precisions and recalls
# nbins = 100 - number of bins used for plotting precision and recall
# clf_inds - indices of sources used for training and testing, 
#            for example, this can be indeces of pulsars, FSRQ, and BLLac sources
#            otherwise, these are indices of all associated sources
# test_frac = 0.3 # fraction of sources used for testing, e.g., 30%


def make_thres_bins(points, values, nbins=100, eps=1.e-15):
    '''
        Calculate average of values at random points inside regular bins
        Input:
            points: 1D locations of values
            values: values to be averaged
            nbins: number of bins
        Output:
            res: average values inside bins where there are non-zero entries
            bin_centers: centers of bins
            mask: array of size nbins, 1 - non-zero number of entries in a bin, 0 - no entries in a bin
                    len(res) = sum(mask)
    '''
    bins = np.linspace(0.-eps, 1.+eps, nbins + 1) # regular grid
    bin_centers = (bins[1:] + bins[:-1])/2 # centers in the regular grid
    nn = np.histogram(points, bins=bins)[0] # number of entries in each bin
    mask = (nn != 0.) # mask that shows bins with non-zero entries
    # average value in each bin where there are non-zero number of entries
    res = np.histogram(points, bins=bins, weights=values)[0][mask] / nn[mask]
    return res, bin_centers, mask


precs_all = np.zeros((nclasses, nav, nbins))
recalls_all = np.zeros((nclasses, nav, nbins))
nnp = np.zeros((nclasses, nav, nbins))
nnr = np.zeros((nclasses, nav, nbins))
class_names = set(ys)

# parameters for the RF classifier
par_dict = {'max_depth': 15, 'n_estimators': 50, 'max_features': "sqrt"}

for j in range(nav):
    
    # separate data into training and testing samples
    rstate = np.random.RandomState(seed=j)
    test_inds = rstate.choice(clf_inds, size=int(len(clf_inds) * test_frac), replace=False, p=None)
    train_inds = [i for i in clf_inds if i not in test_inds]
    X_train = xdf[features].iloc[train_inds]
    X_test = xdf[features].iloc[test_inds]
    y_train = ys[train_inds]
    y_test = ys[test_inds]
    
    # initiate classifier
    clf = RandomForestClassifier(**par_dict)

    # train classifier on the training sample
    clf.fit(X_train, y_train)
    
    # get predictions for the testing sample
    y_pred_test = clf.predict_proba(X_test).T
    
    # one-vs-all precision and recall
    for ind in range(nclasses):
        class_name = class_names[ind]
        clf_ind = list(clf.classes_).index(class_name)
        # clf_ind is in general needed since the order of classes in the classifier clf.classes_
        # is different from class_names
        
        # predicted class probabilities for one class
        y_score_test = y_pred_test[clf_ind]
        y_true_test = (y_test == class_name)
        
        # precision and recall at non-regular grid of probabilities
        prec, recall, thres = precision_recall_curve(y_true_test, y_score_test)
        
        # precision and recall for a regular grid of probability thresholds
        precb, thresc, maskp = make_thres_bins(thres, prec[:-1], nbins=nbins_all)
        recallb, thresc, maskr = make_thres_bins(thres, recall[:-1], nbins=nbins_all)
        
        # save precision, recall and the corresponding positions of non-zero entries
        precs[ind, j][maskp] = precb
        recalls[ind, j][maskr] = recallb
        nnp[ind, j] = maskp.astype(float)
        nnr[ind, j] = maskr.astype(float)

# plot results for the different classes
for ind in range(nclasses):
    maskp = (nnp[ind] > 0)
    mask_avp = (np.sum(nnp[ind], axis=0) > 1) # keep columns with two or more entries
    precs = precs_all[ind, :, mask_avp]
    maskp = maskp[:, mask_avp]
    

    maskr = (nnr[ind] > 0)
    mask_avr = (np.sum(nnr[ind], axis=0) > 1) # keep columns with two or more entries
    recalls = recalls_all[ind, :, mask_avr]
    maskr = maskr[:, mask_avr] 
    

    # average over elements with non-zero number of entries
    threscp = thresc[mask_avp] # grid points with non-zero number of precision entries
    prec_av = np.mean(precs, axis=0, where=maskp)
    prec_std = np.std(precs, axis=0, where=maskp)
    
    threscr = thresc[mask_avr] # grid points with non-zero number of recall entries
    recall_av = np.mean(recalls, axis=0, where=maskr)
    recall_std = np.std(recalls, axis=0, where=maskr)


    pyplot.figure()
    pl = pyplot.plot(threscp, prec_av, label='preicision')
    color = pl[0].get_color()
    pyplot.fill_between(threscp, prec_av - prec_std, prec_av + prec_std, 
                        alpha=0.3, color=color)

    pl = pyplot.plot(threscr, recall_av, ls='--', label='recall')
    color = pl[0].get_color()
    pyplot.fill_between(threscr, recall_av - recall_std, recall_av + recall_std, 
                        alpha=0.3, color=color)
    
    pyplot.legend()
    pyplot.xlim(0.0, 1)
    pyplot.ylim(0.0, 1)
    pyplot.xlabel('Probability threshold')
    pyplot.ylabel('Precision or recall')

    pyplot.title(class_names[ind])
